# Fine-Tuning as a Service on OpenShift AI

## SQL Generation with LoRA Fine-Tuning

This notebook demonstrates **API-driven fine-tuning** on Red Hat OpenShift AI — the same pattern as Google Gemini adapter tuning or Azure OpenAI fine-tuning, but running on **your own infrastructure**.

**Use case**: Fine-tune Qwen2.5-1.5B-Instruct to produce clean, executable SQL from natural language questions.

| Step | What Happens |
|---|---|
| 1 | Prepare training data (SQL question/query pairs) |
| 2 | Evaluate the base model (verbose, unusable output) |
| 3 | Submit a fine-tuning job via `TrainerClient.train()` — **3 lines of Python** |
| 4 | Evaluate the fine-tuned model (bare, executable SQL) |

**Infrastructure**: Training Hub 0.9.2 + Kubeflow Trainer v2 + NVIDIA A10G GPU

---

## 1. Configuration

In [ ]:
import json
import logging
import sys
from pathlib import Path

from datasets import load_dataset
from kubernetes import client as k8s

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)])
for name in ["transformers", "datasets", "torch"]:
    logging.getLogger(name).setLevel(logging.WARNING)

In [ ]:
# Cluster connection — runs inside the workbench pod on OpenShift
api_server = "https://kubernetes.default.svc"

import subprocess
token = subprocess.check_output(
    ["cat", "/var/run/secrets/kubernetes.io/serviceaccount/token"]
).decode().strip()

PVC_NAME = "fine-tuning-shared"
PVC_PATH = "shared"
NAMESPACE = "fine-tuning-demo"

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)

print(f"Connected to: {api_server}")
print(f"Namespace: {NAMESPACE}")
print(f"Shared PVC: {PVC_NAME}")

## 2. Prepare Training Data

We use the public **sql-create-context** dataset: natural language questions paired with SQL queries.

In [ ]:
dataset = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Dataset size: {len(dataset):,} examples")
print(f"Columns: {dataset.column_names}")
print(f"\nSample:")
print(f"  Question: {dataset[0]['question']}")
print(f"  Schema:   {dataset[0]['context'][:80]}...")
print(f"  SQL:      {dataset[0]['answer']}")

In [ ]:
def convert_to_messages(example):
    return {
        "messages": [
            {"role": "user", "content": f"Given the following database schema:\n\n{example['context']}\n\nWrite a SQL query to answer this question: {example['question']}"},
            {"role": "assistant", "content": example["answer"]},
        ]
    }

TRAIN_SIZE = 5000
train_dataset = dataset.shuffle(seed=42).select(range(min(TRAIN_SIZE, len(dataset))))
train_data = [convert_to_messages(ex) for ex in train_dataset]

OUTPUT_DIR = Path(f"/opt/app-root/src/{PVC_PATH}/lora_sql_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_file = OUTPUT_DIR / "train_data.jsonl"

with open(training_file, "w") as f:
    for example in train_data:
        f.write(json.dumps(example) + "\n")

print(f"Training examples: {len(train_data)}")
print(f"Saved to: {training_file}")
print(f"File size: {training_file.stat().st_size / 1024:.1f} KB")

## 3. Baseline Evaluation (BEFORE Fine-Tuning)

First, let's see how the **unmodified base model** handles SQL generation.
The model knows SQL, but wraps it in verbose explanation — unusable for a database API.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading base model: {MODEL_NAME}")
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
base_model.eval()
print(f"Model loaded on: {base_model.device}")

In [ ]:
def generate_sql(model, tokenizer, question, schema, max_tokens=128):
    messages = [{"role": "user", "content": f"Given the following database schema:\n\n{schema}\n\nWrite a SQL query to answer this question: {question}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.1,
                              do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

TEST_EXAMPLES = [
    {"schema": "CREATE TABLE employees (id INT, name VARCHAR, department VARCHAR, salary DECIMAL, hire_date DATE)",
     "question": "What is the average salary of employees in the engineering department?"},
    {"schema": "CREATE TABLE orders (order_id INT, customer_id INT, product_name VARCHAR, quantity INT, order_date DATE)",
     "question": "How many orders were placed in the last 30 days?"},
    {"schema": "CREATE TABLE students (student_id INT, name VARCHAR, grade INT, subject VARCHAR, score DECIMAL)",
     "question": "Find the top 5 students with the highest average score across all subjects."},
]

print("=" * 70)
print("BASELINE: Base Model (BEFORE Fine-Tuning)")
print("=" * 70)
baseline_results = []
for i, ex in enumerate(TEST_EXAMPLES, 1):
    sql = generate_sql(base_model, base_tokenizer, ex["question"], ex["schema"])
    baseline_results.append(sql)
    print(f"\n--- Example {i} ---")
    print(f"Question: {ex['question']}")
    print(f"Output:\n{sql}")
    print()

In [ ]:
del base_model
del base_tokenizer
torch.cuda.empty_cache()
print("Base model unloaded — GPU memory freed for training")

## 4. Submit Fine-Tuning Job via API

This is the key part: **3 lines of Python** to submit a fine-tuning job.

Compare to Google Gemini:
```python
# Google: client.tunings.tune(base_model="gemini-3.5-flash", training_dataset=..., config=...)
# OpenShift AI: client.train(trainer=TrainingHubTrainer(algorithm=LORA_SFT, ...), runtime=th_runtime)
```

Same pattern. Your infrastructure. Your model weights.

In [ ]:
MODEL_PATH = MODEL_NAME
LORA_R = 16
LORA_ALPHA = 32
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 512
MICRO_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 4

params = {
    "model_path": MODEL_PATH,
    "data_path": f"/mnt/{PVC_PATH}/lora_sql_output/train_data.jsonl",
    "ckpt_output_dir": f"/mnt/{PVC_PATH}/checkpoints",
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": 0.0,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "max_seq_len": MAX_SEQ_LEN,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
    "dataset_type": "chat_template",
    "field_messages": "messages",
    "load_in_4bit": True,
    "nproc_per_node": 1,
    "nnodes": 1,
    "logging_steps": 10,
    "save_steps": 200,
    "save_total_limit": 3,
    "save_final_checkpoint": True,
    "lr_scheduler": "cosine",
    "warmup_steps": 0,
    "seed": 42,
}

print("Training Configuration:")
print(f"  Model:      {MODEL_PATH}")
print(f"  Algorithm:  LoRA (rank={LORA_R}, alpha={LORA_ALPHA})")
print(f"  QLoRA:      4-bit quantization enabled")
print(f"  Epochs:     {NUM_EPOCHS}")
print(f"  Batch size: {MICRO_BATCH_SIZE} x {GRADIENT_ACCUMULATION} = {MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  GPU:        1x NVIDIA A10G (24GB)")

In [ ]:
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer
from kubeflow.trainer.options.kubernetes import (
    ContainerOverride, PodSpecOverride, PodTemplateOverride, PodTemplateOverrides,
)

backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)
client = TrainerClient(backend_cfg)

# Find the pre-installed training-hub runtime
th_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == "training-hub":
        th_runtime = runtime
        print(f"Found runtime: {th_runtime.name}")
        break

assert th_runtime is not None, "training-hub ClusterTrainingRuntime not found"

In [ ]:
# === THIS IS THE API CALL — same pattern as Google's client.tunings.tune() ===

job_name = client.train(
    trainer=TrainingHubTrainer(
        algorithm=TrainingHubAlgorithms.LORA_SFT,
        func_args=params,
        env={
            "HF_HOME": f"/mnt/{PVC_PATH}/.cache/huggingface",
        },
        resources_per_node={"cpu": 4, "memory": "24Gi", "nvidia.com/gpu": 1},
    ),
    options=[
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    node_selector={"nvidia.com/gpu.product": "NVIDIA-A10G"},
                    tolerations=[{"effect": "NoSchedule", "key": "nvidia.com/gpu", "operator": "Exists"}],
                    volumes=[
                        {"name": "work", "persistentVolumeClaim": {"claimName": PVC_NAME}},
                        {"name": "dshm", "emptyDir": {"medium": "Memory"}},
                    ],
                    containers=[
                        ContainerOverride(
                            name="node",
                            volume_mounts=[
                                {"name": "work", "mountPath": f"/mnt/{PVC_PATH}", "readOnly": False},
                                {"name": "dshm", "mountPath": "/dev/shm"},
                            ],
                        )
                    ],
                ),
            )
        )
    ],
    runtime=th_runtime,
)

print(f"Training job submitted: {job_name}")
print(f"Monitor in OpenShift console or follow logs below")

In [ ]:
print(f"Following logs for: {job_name}")
print("=" * 70)
logs = client.get_job_logs(job_name, follow=True)
for line in logs:
    print(line)

## 5. Evaluate Fine-Tuned Model (AFTER Fine-Tuning)

Load the LoRA adapter from the shared PVC and test the same prompts.

In [ ]:
import glob
import os

CHECKPOINTS_DIR = f"/opt/app-root/src/{PVC_PATH}/checkpoints"
checkpoint_dirs = sorted(
    glob.glob(os.path.join(CHECKPOINTS_DIR, "checkpoint-*")), key=os.path.getctime
)
CHECKPOINTS_PATH = checkpoint_dirs[-1] if checkpoint_dirs else CHECKPOINTS_DIR
print(f"Loading checkpoint: {CHECKPOINTS_PATH}")

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CHECKPOINTS_PATH, max_seq_length=MAX_SEQ_LEN,
        dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print("Loaded with Unsloth (GPU accelerated)")
except ImportError:
    from peft import PeftModel
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )
    model = PeftModel.from_pretrained(base_model, CHECKPOINTS_PATH)
    model = model.merge_and_unload()
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print("Loaded with HuggingFace/PEFT")

In [ ]:
print("=" * 70)
print("FINE-TUNED: After LoRA Training")
print("=" * 70)
finetuned_results = []
for i, ex in enumerate(TEST_EXAMPLES, 1):
    sql = generate_sql(model, tokenizer, ex["question"], ex["schema"])
    finetuned_results.append(sql)
    print(f"\n--- Example {i} ---")
    print(f"Question: {ex['question']}")
    print(f"Output:\n{sql}")
    print()

## 6. Before / After Comparison

Side-by-side comparison showing the fine-tuning improvement.

In [ ]:
print("\n" + "=" * 70)
print("BEFORE vs AFTER: Side-by-Side Comparison")
print("=" * 70)

for i, ex in enumerate(TEST_EXAMPLES):
    print(f"\n{'─' * 70}")
    print(f"Question: {ex['question']}")
    print(f"{'─' * 70}")
    print(f"\nBEFORE (base model):")
    print(f"  {baseline_results[i][:200]}{'...' if len(baseline_results[i]) > 200 else ''}")
    print(f"\nAFTER (fine-tuned):")
    print(f"  {finetuned_results[i]}")

    is_clean = not any(w in finetuned_results[i].lower() for w in ["here", "query", "following", "use the", "```"])
    print(f"\n  Clean SQL? {'YES' if is_clean else 'NO'}")

print(f"\n{'=' * 70}")
print("SUMMARY")
print(f"{'=' * 70}")
print(f"Base model:     Wraps SQL in explanation and markdown — breaks database APIs")
print(f"Fine-tuned:     Outputs bare executable SQL — ready for production")
print(f"Training time:  ~10 minutes on a single A10G GPU")
print(f"Algorithm:      LoRA (rank={LORA_R}) via Training Hub on OpenShift AI")
print(f"Infrastructure: Your cluster, your data, your model weights")

## 7. Cleanup

In [ ]:
# Uncomment to delete the training job
# client.delete_job(job_name)
# print(f"Deleted training job: {job_name}")